In [ ]:
!pip install -U albumentations ultralytics fiftyone torchview torchinfo torchmetrics 

  Using cached albumentations-2.0.8-py3-none-any.whl.metadata (43 kB)
  Using cached ultralytics-8.4.46-py3-none-any.whl.metadata (39 kB)
  Using cached fiftyone-1.15.0-py3-none-any.whl.metadata (23 kB)
  Using cached torchview-0.2.7-py3-none-any.whl.metadata (13 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached numpy-2.4.4-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached scipy-1.17.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached pydantic-2.13.3-py3-none-any.whl.metadata (108 kB)
  Using cached albucore-0.0.24-py3-none-any.whl.metadata (5.3 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached stringzilla-4.6.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.manyli

# AI/ML Video Analysis Capstone: Advanced Comparative Analysis
This notebook evaluates object detection architectures across multiple datasets. It covers data augmentation, architectural introspection, feature analysis, fine-tuning, and final benchmark evaluations on standard detection datasets like VOC2012 and COCO.

In [ ]:
# --- Utility Modules: Metrics, Video, Evaluation, Analysis, Finetuning, Visualization ---
import matplotlib.pyplot as plt
import cv2, time, torch, numpy as np, os, xml.etree.ElementTree as ET
from torchvision.datasets import VOCDetection
from torchvision.transforms import functional as F
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

VOC_CLASSES = ["background","aeroplane","bicycle","bird","boat","bottle","bus","car","cat","chair","cow","diningtable","dog","horse","motorbike","person","pottedplant","sheep","sofa","train","tvmonitor"]
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(VOC_CLASSES)}

def plot_fps_comparison(results_dict, save_path="output/fps_comparison.png"):
    m, f = list(results_dict.keys()), list(results_dict.values())
    plt.figure(figsize=(8, 5)); plt.bar(m, f, color=['blue','orange','green'][:len(m)])
    plt.ylabel('FPS'); plt.title('Inference Speed'); os.makedirs(os.path.dirname(save_path), exist_ok=True); plt.savefig(save_path); plt.close()

def process_video_stream(model, video_path, out_path, num_frames=30):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened(): return 0
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    s, count = time.time(), 0
    while cap.isOpened() and count < num_frames:
        ret, frame = cap.read()
        if not ret: break
        out.write(model.predict_and_annotate(frame)); count += 1
    cap.release(); out.release()
    return count / (time.time() - s) if count > 0 else 0

def parse_voc_xml(node):
    res = []
    objs = node.get("object", [])
    if not isinstance(objs, list): objs = [objs]
    for o in objs:
        bb = o.get("bndbox")
        if bb: res.append({"name": o.get("name"), "bbox": [int(bb.get("xmin")), int(bb.get("ymin")), int(bb.get("xmax")), int(bb.get("ymax"))]})
    return res

def evaluate_model_on_voc(model_wrapper, dataset_root='./data/voc', num_samples=50):
    dataset = VOCDetection(root=dataset_root, year='2012', image_set='val', download=True)
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    device, model = model_wrapper.device, model_wrapper.model
    model.eval()
    for i in range(min(num_samples, len(dataset))):
        img, target_dict = dataset[i]; objs = parse_voc_xml(target_dict['annotation'])
        gt_b, gt_l = [], []
        for o in objs:
            if o['name'] in CLASS_TO_IDX: gt_b.append(o['bbox']); gt_l.append(CLASS_TO_IDX[o['name']])
        if not gt_b: continue
        target = [dict(boxes=torch.tensor(gt_b, dtype=torch.float32).to(device), labels=torch.tensor(gt_l, dtype=torch.int64).to(device))]
        img_t = F.to_tensor(img).to(device)
        with torch.no_grad():
            if hasattr(model, 'predict'): res = model(img, verbose=False); p = [dict(boxes=res[0].boxes.xyxy.to(device), scores=res[0].boxes.conf.to(device), labels=res[0].boxes.cls.to(torch.int64).to(device))]
            else: preds = model([img_t]); keep = preds[0]['scores'] > 0.1; p = [dict(boxes=preds[0]['boxes'][keep], scores=preds[0]['scores'][keep], labels=preds[0]['labels'][keep])]
        metric.update(p, target)
    r = metric.compute()
    return {'mAP@50': r['map_50'].item(), 'mAP@50-95': r['map'].item(), 'Precision': r['mar_100'].item(), 'Recall': r['mar_1'].item()}

def perform_tsne_pca_analysis(features, labels=None, output_dir="output/feature_analysis"):
    os.makedirs(output_dir, exist_ok=True)
    pca_res = PCA(n_components=2).fit_transform(features)
    plt.figure(figsize=(8,6)); plt.scatter(pca_res[:,0], pca_res[:,1], c=labels, cmap='tab10', alpha=0.7); plt.title('PCA'); plt.savefig(os.path.join(output_dir, "pca.png")); plt.close()
    tsne_res = TSNE(n_components=2, perplexity=min(30, max(5, len(features)//3))).fit_transform(features)
    plt.figure(figsize=(8,6)); plt.scatter(tsne_res[:,0], tsne_res[:,1], c=labels, cmap='tab10', alpha=0.7); plt.title('t-SNE'); plt.savefig(os.path.join(output_dir, "tsne.png")); plt.close()

def finetune_model(model_wrapper, dataset, num_epochs=3, batch_size=4):
    device, model = model_wrapper.device, model_wrapper.model
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=0.005, momentum=0.9, weight_decay=0.0005)
    model.train()
    for epoch in range(num_epochs):
        for imgs, targets in loader:
            imgs = [img.to(device) for img in imgs]; targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss = sum(l for l in model(imgs, targets).values()); optimizer.zero_grad(); loss.backward(); optimizer.step()

class LayerVisualizer:
    def __init__(self, model, output_dir='output/layer_visuals'): self.model, self.output_dir = model, output_dir; os.makedirs(self.output_dir, exist_ok=True); self.activations, self.hooks = {}, []
    def hook_fn(self, name):
        def fn(m, i, o): self.activations[name] = o.detach()
        return fn
    def register_hooks(self, layers):
        for n, m in self.model.named_modules():
            if n in layers: self.hooks.append(m.register_forward_hook(self.hook_fn(n)))
    def remove_hooks(self):
        for h in self.hooks: h.remove()
        self.hooks = []
    def plot_architecture(self, filename='arch'):
        try:
            from torchview import draw_graph
            g = draw_graph(self.model, input_size=(1,3,640,640), expand_nested=True, depth=10, device='cpu')
            g.visual_graph.render(os.path.join(self.output_dir, filename), format='png', cleanup=True)
            return os.path.join(self.output_dir, filename) + '.png'
        except: return None


In [ ]:
# --- Data Engineering: Augmentations & Dataset Loader ---
import os, glob, cv2, numpy as np, urllib.request, zipfile
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, ConcatDataset
from torchvision.datasets import VOCDetection

COCO_CLASSES = ['person','bicycle','car','motorcycle','airplane','bus','train','truck','boat','traffic light','fire hydrant','stop sign','parking meter','bench','bird','cat','dog','horse','sheep','cow','elephant','bear','zebra','giraffe','backpack','umbrella','handbag','tie','suitcase','frisbee','skis','snowboard','sports ball','kite','baseball bat','baseball glove','skateboard','surfboard','tennis racket','bottle','wine glass','cup','fork','knife','spoon','bowl','banana','apple','sandwich','orange','broccoli','carrot','hot dog','pizza','donut','cake','chair','couch','potted plant','bed','dining table','toilet','tv','laptop','mouse','remote','keyboard','cell phone','microwave','oven','toaster','sink','refrigerator','book','clock','vase','scissors','teddy bear','hair drier','toothbrush']
VOC_CLASSES = ['background','aeroplane','bicycle','bird','boat','bottle','bus','car','cat','chair','cow','diningtable','dog','horse','motorbike','person','pottedplant','sheep','sofa','train','tvmonitor']

def get_transforms(train=True):
    t = [A.LongestMaxSize(max_size=640), A.PadIfNeeded(640, 640, border_mode=cv2.BORDER_CONSTANT, value=114)]
    if train: t += [A.HorizontalFlip(p=0.5), A.RandomBrightnessContrast(p=0.5), A.GaussNoise(p=0.2)]
    t += [A.Normalize(), ToTensorV2()]
    return A.Compose(t, bbox_params=A.BboxParams(format='pascal_voc', label_fields=['class_labels'], clip=True))

def get_train_transforms(): return get_transforms(True)
def get_val_transforms(): return get_transforms(False)

class VOCDatasetWrapper(Dataset):
    def __init__(self, ds, tf=None): self.ds, self.tf, self.c2i = ds, tf, {c: i for i, c in enumerate(VOC_CLASSES)}
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        img, ann = self.ds[i]; img = np.array(img); h, w = img.shape[:2]; boxes, labels = [], []
        objs = ann['annotation'].get('object', []); objs = [objs] if not isinstance(objs, list) else objs
        for o in objs:
            b = o['bndbox']; x1, y1, x2, y2 = max(0, int(float(b['xmin']))), max(0, int(float(b['ymin']))), min(w, int(float(b['xmax']))), min(h, int(float(b['ymax'])))
            if x2 > x1 and y2 > y1: boxes.append([x1, y1, x2, y2]); labels.append(self.c2i.get(o['name'], 0))
        if not boxes: boxes, labels = [[0,0,1,1]], [0]
        if self.tf: res = self.tf(image=img, bboxes=boxes, class_labels=labels); img, boxes, labels = res['image'], list(res['bboxes']), list(res['class_labels'])
        return img, boxes, labels

class YOLODatasetWrapper(Dataset):
    def __init__(self, i_dir, l_dir, tf=None):
        self.tf, self.samples = tf, []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            for ip in glob.glob(os.path.join(i_dir, '**', ext), recursive=True):
                lp = os.path.join(l_dir, os.path.splitext(os.path.basename(ip))[0] + '.txt')
                if not os.path.exists(lp):
                    ms = glob.glob(os.path.join(l_dir, '**', os.path.basename(lp)), recursive=True)
                    lp = ms[0] if ms else None
                self.samples.append((ip, lp))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        ip, lp = self.samples[i]; img = cv2.cvtColor(cv2.imread(ip), cv2.COLOR_BGR2RGB); h, w = img.shape[:2]; boxes, labels = [], []
        if lp and os.path.exists(lp):
            with open(lp) as f:
                for line in f:
                    p = line.strip().split()
                    if len(p) < 5: continue
                    cls, cx, cy, bw, bh = int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4])
                    x1, y1, x2, y2 = max(0, int((cx-bw/2)*w)), max(0, int((cy-bh/2)*h)), min(w, int((cx+bw/2)*w)), min(h, int((cy+bh/2)*h))
                    if x2 > x1 and y2 > y1: boxes.append([x1, y1, x2, y2]); labels.append(cls)
        if not boxes: boxes, labels = [[0,0,1,1]], [0]
        if self.tf: res = self.tf(image=img, bboxes=boxes, class_labels=labels); img, boxes, labels = res['image'], list(res['bboxes']), list(res['class_labels'])
        return img, boxes, labels

class DatasetLoader:
    def __init__(self, data_dir="data"): self.data_dir = data_dir; os.makedirs(self.data_dir, exist_ok=True)
    def load_voc(self, year='2012'): return VOCDetection(root=os.path.join(self.data_dir, 'voc'), year=year, image_set='val', download=True)
    def load_coco128(self):
        url, fp, ed = "https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip", os.path.join(self.data_dir, "coco128.zip"), os.path.join(self.data_dir, "coco128")
        if not os.path.exists(ed):
             if not os.path.exists(fp): urllib.request.urlretrieve(url, fp)
             with zipfile.ZipFile(fp, 'r') as z: z.extractall(self.data_dir)
        return ed
    def load_coco8(self):
        url, fp, ed = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco8.zip", os.path.join(self.data_dir, "coco8.zip"), os.path.join(self.data_dir, "coco8")
        if not os.path.exists(ed):
             if not os.path.exists(fp): urllib.request.urlretrieve(url, fp)
             with zipfile.ZipFile(fp, 'r') as z: z.extractall(self.data_dir)
        return ed
    def load_open_images(self, max_samples=500):
        try:
            import fiftyone as fo, fiftyone.zoo as foz
            fo.config.dataset_zoo_dir = os.path.join(self.data_dir, "open_images")
            return foz.load_zoo_dataset("open-images-v7", split="validation", max_samples=max_samples, drop_existing_dataset=True)
        except: return None


In [ ]:
# --- Model Architectures: Base, YOLO, RCNN, SSD, RetinaNet, FCOS ---
import torch, torch.nn as nn, cv2
from abc import ABC, abstractmethod
from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights, ssd300_vgg16, SSD300_VGG16_Weights, retinanet_resnet50_fpn_v2, RetinaNet_ResNet50_FPN_V2_Weights, fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
from torchvision.transforms import functional as F

class BaseModel(ABC):
    def __init__(self, device): self.device, self.model = device, None
    @abstractmethod
    def load(self): pass
    @abstractmethod
    def predict_and_annotate(self, frame): pass

class YOLOModel(BaseModel):
    def __init__(self, device, weights_path='yolo11n.pt'): super().__init__(device); self.weights_path = weights_path
    def load(self): self.model = YOLO(self.weights_path); self.model.to(self.device)
    def predict_and_annotate(self, frame): return self.model(frame, verbose=False, device=self.device)[0].plot()

class CustomBoxPredictor(nn.Module):
    def __init__(self, in_f, num_c):
        super().__init__()
        self.cls_score = nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Dropout(0.3), nn.Linear(512, num_c))
        self.bbox_pred = nn.Sequential(nn.Linear(in_f, 512), nn.ReLU(), nn.Linear(512, num_c * 4))
    def forward(self, x): x = x.flatten(start_dim=1); return self.cls_score(x), self.bbox_pred(x)

class CustomRCNNModel(BaseModel):
    def load(self):
        self.model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
        in_f = self.model.roi_heads.box_predictor.cls_score.in_features
        self.model.roi_heads.box_predictor = CustomBoxPredictor(in_f, 91)
        self.model.to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        img_t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([img_t])[0]
        ann = frame.copy()
        for i in range(len(pred['boxes'])):
            if pred['scores'][i] > 0.5:
                b = pred['boxes'][i].cpu().numpy().astype(int); cv2.rectangle(ann, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
        return ann

class SSDModel(BaseModel):
    def load(self): self.model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT); self.model.to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        img_t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([img_t])[0]
        ann = frame.copy()
        for i in range(len(pred['boxes'])):
            if pred['scores'][i] > 0.5:
                b = pred['boxes'][i].cpu().numpy().astype(int); cv2.rectangle(ann, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
        return ann

class RetinaNetModel(BaseModel):
    def load(self): self.model = retinanet_resnet50_fpn_v2(weights=RetinaNet_ResNet50_FPN_V2_Weights.DEFAULT); self.model.to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        img_t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([img_t])[0]
        ann = frame.copy()
        for i in range(len(pred['boxes'])):
            if pred['scores'][i] > 0.5:
                b = pred['boxes'][i].cpu().numpy().astype(int); cv2.rectangle(ann, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
        return ann

class FCOSModel(BaseModel):
    def load(self): self.model = fcos_resnet50_fpn(weights=FCOS_ResNet50_FPN_Weights.DEFAULT); self.model.to(self.device); self.model.eval()
    def predict_and_annotate(self, frame):
        img_t = F.to_tensor(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)).to(self.device)
        with torch.no_grad(): pred = self.model([img_t])[0]
        ann = frame.copy()
        for i in range(len(pred['boxes'])):
            if pred['scores'][i] > 0.5:
                b = pred['boxes'][i].cpu().numpy().astype(int); cv2.rectangle(ann, (b[0], b[1]), (b[2], b[3]), (0, 255, 0), 2)
        return ann


In [ ]:
# --- Module Bridge: Registering virtual modules for existing imports ---
import sys, types
def reg(p, m):
    pts = p.split('.')
    for i in range(len(pts)):
        s = '.'.join(pts[:i+1])
        if s not in sys.modules: sys.modules[s] = types.ModuleType(s)
    mod = sys.modules[p]
    for k, v in m.items(): setattr(mod, k, v)

reg('models.base_model', {'BaseModel': BaseModel})
reg('models.yolo_model', {'YOLOModel': YOLOModel})
reg('models.custom_rcnn', {'CustomRCNNModel': CustomRCNNModel, 'CustomBoxPredictor': CustomBoxPredictor})
reg('models.ssd_model', {'SSDModel': SSDModel})
reg('models.retinanet_model', {'RetinaNetModel': RetinaNetModel})
reg('models.fcos_model', {'FCOSModel': FCOSModel})
reg('utils.metrics', {'plot_fps_comparison': plot_fps_comparison})
reg('utils.video_utils', {'process_video_stream': process_video_stream})
reg('utils.evaluation', {'evaluate_model_on_voc': evaluate_model_on_voc, 'parse_voc_xml': parse_voc_xml, 'CLASS_TO_IDX': CLASS_TO_IDX})
reg('utils.feature_analysis', {'perform_tsne_pca_analysis': perform_tsne_pca_analysis})
reg('utils.finetuning', {'finetune_model': finetune_model})
reg('utils.layer_visualizer', {'LayerVisualizer': LayerVisualizer})
reg('data.augmentations', {'get_train_transforms': get_train_transforms, 'get_val_transforms': get_val_transforms, 'VOCDatasetWrapper': VOCDatasetWrapper, 'YOLODatasetWrapper': YOLODatasetWrapper, 'COCO_CLASSES': COCO_CLASSES, 'VOC_CLASSES': VOC_CLASSES})
reg('data.dataset_loader', {'DatasetLoader': DatasetLoader})
print("✓ Module Bridge initialized.")


In [ ]:
import torch
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F
import torch.nn as nn

from models.yolo_model import YOLOModel
from models.custom_rcnn import CustomRCNNModel
from models.ssd_model import SSDModel
from models.retinanet_model import RetinaNetModel
from models.fcos_model import FCOSModel
from utils.video_utils import process_video_stream
from utils.evaluation import evaluate_model_on_voc, parse_voc_xml, CLASS_TO_IDX
from data.dataset_loader import DatasetLoader
from data.augmentations import get_train_transforms
from utils.layer_visualizer import LayerVisualizer
from utils.feature_analysis import perform_tsne_pca_analysis
from utils.finetuning import finetune_model

# Determine device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on: {device}")


## 1. Dataset Integration & Augmentation
Loading standard benchmark object detection datasets (VOC2012, COCO128, COCO8, Open Images v7)
and visualising sample images with bounding boxes, segmentation info, and raw annotation values.

In [ ]:
loader = DatasetLoader(data_dir='data')
print("Initializing dataset downloads and loaders...")

voc2012     = loader.load_voc('2012')
coco128_dir = loader.load_coco128()
coco8_dir   = loader.load_coco8()
open_images = loader.load_open_images(max_samples=500)

print(f"\nVOC 2012 : {len(voc2012)} images")
print(f"COCO128  : {coco128_dir}")
print(f"COCO8    : {coco8_dir}")
print(f"Open Img : {len(open_images) if open_images else 'N/A'} images")


### 1a. Dataset Sample Visualisation
Showing 4 sample images per dataset with bounding boxes overlaid and raw annotation values printed.

In [ ]:
import os, glob, cv2, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from data.augmentations import COCO_CLASSES, VOC_CLASSES

def draw_boxes(ax, image, boxes, labels, class_names, title=""):
    ax.imshow(image)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.axis('off')
    colors = plt.cm.Set2.colors
    for (x1, y1, x2, y2), lbl in zip(boxes, labels):
        color = colors[int(lbl) % len(colors)]
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor=color, facecolor='none')
        ax.add_patch(rect)
        cname = class_names[int(lbl)] if int(lbl) < len(class_names) else str(int(lbl))
        ax.text(x1, y1-4, cname, fontsize=7, color='white',
                bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor='none'))

# ── VOC 2012 ─────────────────────────────────────────────────────────────────
print("=" * 60)
print("VOC 2012  |  bbox format: [xmin ymin xmax ymax] (absolute px)")
print("=" * 60)
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle("VOC 2012 — Samples with Bounding Boxes", fontsize=13, fontweight='bold')
for i, ax in enumerate(axes):
    pil_img, ann = voc2012[i]
    img = np.array(pil_img)
    objs = ann['annotation'].get('object', [])
    if not isinstance(objs, list): objs = [objs]
    boxes, labels = [], []
    print(f"\n  Image {i}: {ann['annotation']['filename']}  size={img.shape[:2]}")
    for obj in objs:
        bb = obj['bndbox']
        x1,y1,x2,y2 = int(float(bb['xmin'])),int(float(bb['ymin'])),int(float(bb['xmax'])),int(float(bb['ymax']))
        lbl = VOC_CLASSES.index(obj['name']) if obj['name'] in VOC_CLASSES else 0
        boxes.append([x1,y1,x2,y2]); labels.append(lbl)
        print(f"    {obj['name']:15s} bbox=[{x1:4d},{y1:4d},{x2:4d},{y2:4d}]  difficult={obj.get('difficult','0')}")
    draw_boxes(ax, img, boxes, labels, VOC_CLASSES, f"VOC #{i}")
plt.tight_layout(); plt.show()

# ── COCO8 ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("COCO8  |  label format: class cx cy w h (normalised 0-1)")
print("=" * 60)
coco8_imgs = sorted(glob.glob(os.path.join(coco8_dir, 'images', 'train', '*.jpg')))[:4]
fig, axes = plt.subplots(1, len(coco8_imgs), figsize=(18, 5))
fig.suptitle("COCO8 — Samples with Bounding Boxes", fontsize=13, fontweight='bold')
for ax, img_path in zip(axes, coco8_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(coco8_dir, 'labels', 'train', stem + '.txt')
    boxes, labels = [], []
    print(f"\n  {os.path.basename(img_path)}  ({w}x{h})")
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                cls,cx,cy,bw,bh = int(p[0]),float(p[1]),float(p[2]),float(p[3]),float(p[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                boxes.append([x1,y1,x2,y2]); labels.append(cls)
                cname = COCO_CLASSES[cls] if cls < len(COCO_CLASSES) else str(cls)
                print(f"    {cname:20s} raw=[{cls} {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f}]  abs=[{x1},{y1},{x2},{y2}]")
    draw_boxes(ax, img, boxes, labels, COCO_CLASSES, os.path.basename(img_path))
plt.tight_layout(); plt.show()

# ── COCO128 ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("COCO128  |  label format: class cx cy w h (normalised 0-1)")
print("=" * 60)
c128_imgs = sorted(glob.glob(os.path.join(coco128_dir, 'images', 'train2017', '*.jpg')))[:4]
fig, axes = plt.subplots(1, len(c128_imgs), figsize=(18, 5))
fig.suptitle("COCO128 — Samples with Bounding Boxes", fontsize=13, fontweight='bold')
for ax, img_path in zip(axes, c128_imgs):
    img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = os.path.join(coco128_dir, 'labels', 'train2017', stem + '.txt')
    boxes, labels = [], []
    print(f"\n  {os.path.basename(img_path)}  ({w}x{h})")
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                cls,cx,cy,bw,bh = int(p[0]),float(p[1]),float(p[2]),float(p[3]),float(p[4])
                x1=int((cx-bw/2)*w); y1=int((cy-bh/2)*h)
                x2=int((cx+bw/2)*w); y2=int((cy+bh/2)*h)
                boxes.append([x1,y1,x2,y2]); labels.append(cls)
                cname = COCO_CLASSES[cls] if cls < len(COCO_CLASSES) else str(cls)
                print(f"    {cname:20s} raw=[{cls} {cx:.4f} {cy:.4f} {bw:.4f} {bh:.4f}]  abs=[{x1},{y1},{x2},{y2}]")
    draw_boxes(ax, img, boxes, labels, COCO_CLASSES, os.path.basename(img_path))
plt.tight_layout(); plt.show()

# ── Open Images v7 ────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("Open Images v7  |  bbox: [top-left-x top-left-y width height] (normalised)")
print("=" * 60)
if open_images:
    oi_samples = list(open_images.take(4))
    fig, axes = plt.subplots(1, len(oi_samples), figsize=(18, 5))
    fig.suptitle("Open Images v7 — Samples with Bounding Boxes", fontsize=13, fontweight='bold')
    for ax, sample in zip(axes, oi_samples):
        img = cv2.cvtColor(cv2.imread(sample.filepath), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        boxes, labels = [], []
        dets = sample.get_field('detections')
        print(f"\n  {os.path.basename(sample.filepath)}  ({w}x{h})")
        if dets and hasattr(dets, 'detections'):
            for det in dets.detections[:6]:
                bx,by,bw,bh = det.bounding_box
                x1=int(bx*w); y1=int(by*h); x2=int((bx+bw)*w); y2=int((by+bh)*h)
                boxes.append([x1,y1,x2,y2]); labels.append(0)
                print(f"    {det.label:25s} normalised=[{bx:.4f},{by:.4f},{bw:.4f},{bh:.4f}]  abs=[{x1},{y1},{x2},{y2}]")
        draw_boxes(ax, img, boxes, labels, [det.label for det in (dets.detections if dets else [])], os.path.basename(sample.filepath))
    plt.tight_layout(); plt.show()
else:
    print("Open Images not loaded — skipping.")


### 1b. Augmentation Preview — Before vs After
Showing the same sample from each dataset before and after the full training augmentation pipeline (resize→flip→colour jitter→noise→normalise).

In [ ]:
import torch
import albumentations as A
from data.augmentations import (get_train_transforms, get_val_transforms,
                                 VOCDatasetWrapper, YOLODatasetWrapper,
                                 OpenImagesDatasetWrapper, COCO_CLASSES, VOC_CLASSES)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def tensor_to_rgb(t):
    """Undo ImageNet normalisation for display."""
    img = t.permute(1, 2, 0).numpy()
    img = img * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

train_tf = get_train_transforms()

# Collect one raw sample from each dataset
raw_samples = []

# VOC
pil_img, ann = voc2012[0]
objs = ann['annotation'].get('object', [])
if not isinstance(objs, list): objs = [objs]
voc_boxes, voc_labels = [], []
for obj in objs:
    bb = obj['bndbox']
    x1,y1,x2,y2 = int(float(bb['xmin'])),int(float(bb['ymin'])),int(float(bb['xmax'])),int(float(bb['ymax']))
    lbl = VOC_CLASSES.index(obj['name']) if obj['name'] in VOC_CLASSES else 0
    voc_boxes.append([x1,y1,x2,y2]); voc_labels.append(lbl)
raw_samples.append(("VOC 2012", np.array(pil_img), voc_boxes, voc_labels, VOC_CLASSES))

# COCO8
img_path = sorted(glob.glob(os.path.join(coco8_dir,'images','train','*.jpg')))[0]
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
h,w = img.shape[:2]
lbl_path = os.path.join(coco8_dir,'labels','train', os.path.splitext(os.path.basename(img_path))[0]+'.txt')
c8_boxes, c8_labels = [], []
if os.path.exists(lbl_path):
    for line in open(lbl_path):
        p=line.strip().split(); cls,cx,cy,bw,bh=int(p[0]),float(p[1]),float(p[2]),float(p[3]),float(p[4])
        c8_boxes.append([int((cx-bw/2)*w),int((cy-bh/2)*h),int((cx+bw/2)*w),int((cy+bh/2)*h)]); c8_labels.append(cls)
raw_samples.append(("COCO8", img, c8_boxes, c8_labels, COCO_CLASSES))

# COCO128
img_path = sorted(glob.glob(os.path.join(coco128_dir,'images','train2017','*.jpg')))[0]
img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
h,w = img.shape[:2]
lbl_path = os.path.join(coco128_dir,'labels','train2017', os.path.splitext(os.path.basename(img_path))[0]+'.txt')
c12_boxes, c12_labels = [], []
if os.path.exists(lbl_path):
    for line in open(lbl_path):
        p=line.strip().split(); cls,cx,cy,bw,bh=int(p[0]),float(p[1]),float(p[2]),float(p[3]),float(p[4])
        c12_boxes.append([int((cx-bw/2)*w),int((cy-bh/2)*h),int((cx+bw/2)*w),int((cy+bh/2)*h)]); c12_labels.append(cls)
raw_samples.append(("COCO128", img, c12_boxes, c12_labels, COCO_CLASSES))

# Open Images
if open_images:
    s = list(open_images.take(1))[0]
    img = cv2.cvtColor(cv2.imread(s.filepath), cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    oi_boxes, oi_labels = [], []
    dets = s.get_field('detections')
    if dets and hasattr(dets,'detections'):
        for det in dets.detections[:5]:
            bx,by,bw_n,bh_n = det.bounding_box
            oi_boxes.append([int(bx*w),int(by*h),int((bx+bw_n)*w),int((by+bh_n)*h)])
            oi_labels.append(0)
    raw_samples.append(("Open Images v7", img, oi_boxes, oi_labels, COCO_CLASSES))

# ── Plot before / after ──────────────────────────────────────────────────────
n = len(raw_samples)
fig, axes = plt.subplots(n, 2, figsize=(14, 5 * n))
fig.suptitle("Augmentation Preview — Before vs After (640×640 letterbox)", fontsize=14, fontweight='bold')

for row, (name, raw_img, boxes, labels, cls_names) in enumerate(raw_samples):
    # Before
    ax_before = axes[row, 0]
    ax_before.set_title(f"{name} — Original", fontsize=10)
    draw_boxes(ax_before, raw_img, boxes, labels, cls_names)

    # After augmentation
    boxes_valid = [b for b in boxes if b[2]>b[0] and b[3]>b[1]] or [[0,0,1,1]]
    labels_valid = labels[:len(boxes_valid)] if labels else [0]
    result = train_tf(image=raw_img.copy(), bboxes=boxes_valid, class_labels=labels_valid)
    aug_img  = tensor_to_rgb(result['image'])
    aug_boxes  = [[int(x) for x in b] for b in result['bboxes']]
    aug_labels = list(result['class_labels'])

    ax_after = axes[row, 1]
    ax_after.set_title(f"{name} — Augmented (640×640)", fontsize=10)
    draw_boxes(ax_after, aug_img, aug_boxes, aug_labels, cls_names)

    print(f"[{name}] original={raw_img.shape[:2]}  →  augmented=640×640  boxes kept={len(aug_boxes)}")

plt.tight_layout(); plt.show()


### 1c. Combined Dataset → DataLoader for Models
All four datasets are merged into a single `ConcatDataset` with the full training augmentation pipeline. A `DataLoader` is constructed and used to feed batches to each model.

In [ ]:
from torch.utils.data import DataLoader
from data.augmentations import build_combined_dataset
import torch

def collate_fn(batch):
    """Custom collate: images stacked, boxes/labels kept as lists."""
    images  = torch.stack([b[0] for b in batch])
    boxes   = [b[1] for b in batch]
    labels  = [b[2] for b in batch]
    return images, boxes, labels

print("Building combined dataset from all sources...")
combined_ds = build_combined_dataset(
    voc_dataset        = voc2012,
    coco128_dir        = coco128_dir,
    coco8_dir          = coco8_dir,
    open_images_dataset= open_images,
    mode               = 'train',
)

combined_loader = DataLoader(
    combined_ds,
    batch_size  = 4,
    shuffle     = True,
    num_workers = 2,
    collate_fn  = collate_fn,
)

# Sanity-check: pull one batch and print shapes
images, boxes, labels = next(iter(combined_loader))
print(f"\nSample batch — images tensor : {images.shape}")
print(f"               boxes (list)  : {[len(b) for b in boxes]} objects per image")
print(f"               labels (list) : {[len(l) for l in labels]} labels  per image")
print("\n✓ DataLoader ready. Batches flow directly into model forward passes below.")


## 2. Load Models & Plot Architectures
Initializing models, printing their CNN layer weights, and plotting architecture diagrams.

## 3. Real Feature Extraction Analysis (t-SNE & PCA)
Passing real VOC images through the network to extract bottleneck features and project them into 2D space.

In [ ]:
# Extract real features from Custom Faster R-CNN backbone using 50 real VOC images
rcnn_model = models['Custom Faster R-CNN'].model
visualizer = LayerVisualizer(rcnn_model)

# Find a deep CNN layer to hook into
target_layer = 'backbone.body.layer4'
visualizer.register_hooks([target_layer])

extracted_features = []
labels = []
print("Extracting real features from 50 VOC2012 images...")

rcnn_model.eval()
with torch.no_grad():
    for i in range(50):
        img, target_dict = voc2012[i]
        img_tensor = F.to_tensor(img).unsqueeze(0).to(device)
        
        # Forward pass to trigger hook
        try:
             rcnn_model(img_tensor)
             act = visualizer.activations[target_layer]
             # Global Average Pooling to get a 1D feature vector
             feat = act.mean(dim=[2,3]).squeeze(0).cpu().numpy()
             extracted_features.append(feat)
             
             # Extract simple label (first object in image)
             objects = target_dict['annotation'].get('object', [])
             if not isinstance(objects, list):
                 objects = [objects]
             lbl = objects[0]['name'] if objects else 'background'
             labels.append(lbl)
        except Exception as e:
             continue

visualizer.remove_hooks()

if extracted_features:
    extracted_features = np.array(extracted_features)
    # Convert string labels to ints for plotting
    unique_labels = list(set(labels))
    int_labels = [unique_labels.index(l) for l in labels]
    
    print(f"Extracted feature matrix of shape: {extracted_features.shape}")
    perform_tsne_pca_analysis(extracted_features, int_labels)
else:
    print("Failed to extract features.")


## 4. Fine-Tuning Pipeline
Fine-tuning the network on a targeted subset of real images.

In [ ]:
# Create a dataset wrapper to convert VOC format into PyTorch tensor targets
class FinetuneDataset(torch.utils.data.Dataset):
    def __init__(self, voc_dataset, max_samples=10):
        self.voc = voc_dataset
        self.max_samples = min(max_samples, len(voc_dataset))
        
    def __len__(self):
        return self.max_samples
        
    def __getitem__(self, idx):
        img, target_dict = self.voc[idx]
        img_tensor = F.to_tensor(img)
        
        objects = parse_voc_xml(target_dict['annotation'])
        gt_boxes, gt_labels = [], []
        for obj in objects:
            if obj['name'] in CLASS_TO_IDX:
                gt_boxes.append(obj['bbox'])
                gt_labels.append(CLASS_TO_IDX[obj['name']])
        
        if len(gt_boxes) == 0:
            gt_boxes = [[0, 0, 1, 1]] # Fallback
            gt_labels = [0]
            
        target = {
            'boxes': torch.tensor(gt_boxes, dtype=torch.float32),
            'labels': torch.tensor(gt_labels, dtype=torch.int64)
        }
        return img_tensor, target

ft_dataset = FinetuneDataset(voc2012, max_samples=8)

# Finetuning the Custom Faster R-CNN on the subset
print("Starting finetuning loop on Custom Faster R-CNN...")
finetune_model(models['Custom Faster R-CNN'], ft_dataset, num_epochs=2, batch_size=2)
print("Finetuning pipeline executed successfully.")


## 5. Model Accuracy Evaluation
Calculating Mean Average Precision (mAP), Precision, and Recall using `torchmetrics` on the validation subset (VOC2012).

In [ ]:
accuracy_results = {}
for name, model in models.items():
    res = evaluate_model_on_voc(model, num_samples=30)
    accuracy_results[name] = res

df_acc = pd.DataFrame(accuracy_results).T
display(df_acc)


## 6. Inference Speed Benchmark (FPS)
Measuring real-time throughput on a live pedestrian video stream.

In [ ]:
loader = DatasetLoader()
video_path = loader.download_sample_video()

fps_results = {}
for name, model in models.items():
    out_path = f"output/{name.replace(' ', '_')}_out.mp4"
    fps = process_video_stream(model, video_path, out_path, num_frames=30)
    fps_results[name] = fps

df_fps = pd.DataFrame.from_dict(fps_results, orient='index', columns=['FPS'])
display(df_fps)


## 7. Final Comparative Visualizations
Plotting accuracy vs throughput.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Accuracy Plot
df_acc['mAP@50'].plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Accuracy (mAP@50)')
axes[0].set_ylabel('mAP')
axes[0].tick_params(axis='x', rotation=45)

# Speed Plot
df_fps['FPS'].plot(kind='bar', ax=axes[1], color='salmon')
axes[1].set_title(f'Inference Speed (FPS) on {device.type.upper()}')
axes[1].set_ylabel('Frames Per Second')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
